# Phase 2: Feature Engineering

From cleaned.csv to model_ready.csv. Encode bins, derive composite features, build the discomfort target.

## Step 0 — Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)

CWD = Path.cwd()
ROOT = CWD if (CWD / 'data').exists() else CWD.parent
df = pd.read_csv(ROOT / 'data' / 'processed' / 'cleaned.csv')
print(df.shape)

(182, 48)


## Step 1 — Inspect dtypes and remaining text columns

In [2]:
df.dtypes.value_counts()

str      36
int64    12
Name: count, dtype: int64

In [3]:
text_cols = df.select_dtypes(include='object').columns.tolist()
print(len(text_cols), 'text columns')
text_cols

36 text columns


C:\Users\ailav\AppData\Local\Temp\ipykernel_19432\3505164713.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include='object').columns.tolist()


['Timestamp',
 'Gender',
 'Age',
 'Education',
 'Region',
 'Marital status',
 'Delivery platform',
 'Job duration (experience)',
 'Monthly income',
 'Type of vehicle',
 'Mode of carrying deliveries',
 'Working Hours per Day',
 'Working days per week',
 'Number of deliveries per day',
 'Duration of rest break',
 'Type of break',
 'Tobacco Consumption',
 'Alcohol Consumption',
 'In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Row 1Neck]',
 'In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Shoulders]',
 'In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Upper back]',
 'In the past 12 months, have you experienced pain, discomfort, or numbness in the following body areas due to delivery work? [Lower back]',
 'In the past 12 months, have you experienced pain, discomfo

## Step 2 — Ordinal-encode the ordered bins

In [4]:
ordinal_maps = {
    'Age': {'<25': 0, '25-35': 1, '36-45': 2, '>=46': 3},
    'Education': {'Middle school or primary school': 0, 'High school': 1, "Degree/master's degree or higher": 2},
    'Monthly income': {'<10,000': 0, '10,000 - 19,999': 1, '20,000 - 35,000': 2, '>35,000': 3},
    'Job duration (experience)': {'<6 months': 0, '6 -12 months': 1, '12 - 24 months': 2, '>24 months': 3},
    'Working Hours per Day': {'< 3 hrs': 0, '3-5 hrs': 1, '6-8 hrs': 2, '> 8 hrs': 3},
    'Working days per week': {'< 3': 0, '3-4': 1, '5-6': 2, '7': 3},
    'Number of deliveries per day': {'<=10': 0, '11-20': 1, '21-30': 2, '>=31': 3},
    'Duration of rest break': {'<5 min': 0, '5-15 min': 1, '16-30min': 2, '>30 min': 3},
}

ord_names = {
    'Age': 'age_ord',
    'Education': 'education_ord',
    'Monthly income': 'income_ord',
    'Job duration (experience)': 'job_duration_ord',
    'Working Hours per Day': 'work_hours_ord',
    'Working days per week': 'work_days_ord',
    'Number of deliveries per day': 'deliveries_ord',
    'Duration of rest break': 'rest_break_ord',
}

for col, mapping in ordinal_maps.items():
    df[ord_names[col]] = df[col].map(mapping)
    assert df[ord_names[col]].notna().all(), col

df[list(ord_names.values())].head()

,age_ord,education_ord,income_ord,job_duration_ord,work_hours_ord,work_days_ord,deliveries_ord,rest_break_ord
0,1,2,1,0,1,3,1,1
1,1,1,1,1,2,3,2,3
2,0,2,2,0,3,3,3,3
3,0,1,0,1,1,1,1,2
4,3,1,1,1,2,1,1,2


## Step 3 — Numeric midpoints for work-exposure bins

In [5]:
# Midpoints feed the Phase 3 risk formulas (e.g. deliveries / hours)
midpoint_maps = {
    'Working Hours per Day':        ({'< 3 hrs': 2, '3-5 hrs': 4, '6-8 hrs': 7, '> 8 hrs': 9}, 'work_hours_num'),
    'Number of deliveries per day': ({'<=10': 5, '11-20': 15, '21-30': 25, '>=31': 35}, 'deliveries_num'),
    'Working days per week':        ({'< 3': 2, '3-4': 3.5, '5-6': 5.5, '7': 7}, 'work_days_num'),
    'Duration of rest break':       ({'<5 min': 2.5, '5-15 min': 10, '16-30min': 23, '>30 min': 40}, 'rest_break_num'),
}

for col, (mapping, name) in midpoint_maps.items():
    df[name] = df[col].map(mapping)
    assert df[name].notna().all(), col

df[[v[1] for v in midpoint_maps.values()]].head()

,work_hours_num,deliveries_num,work_days_num,rest_break_num
0,4,15,7.0,10.0
1,7,25,7.0,40.0
2,9,35,7.0,40.0
3,4,15,3.5,23.0
4,7,15,3.5,23.0


## Step 4 — Binary-encode the Yes/No items

In [6]:
# Nordic pain (9), 7-day discomfort (4), plus the other Yes/No items
yesno_cols = [c for c in df.columns
              if df[c].dtype == object and set(df[c].dropna().unique()) <= {'Yes', 'No'}]
print(len(yesno_cols), 'Yes/No columns')

for c in yesno_cols:
    df[c] = (df[c] == 'Yes').astype(int)

df[yesno_cols].head()

0 Yes/No columns


""
0
1
2
3
4


## Step 5 — Rank vehicle and carrying mode

In [7]:
# vehicle: motorbike vibrates more than scooter
df['vehicle_rank'] = df['Type of vehicle'].map({'Scooter': 1, 'Motor bike': 2})

# carrying: storage box (least body contact) -> handheld (most)
df['carrying_contact_rank'] = df['Mode of carrying deliveries'].map(
    {'Bike storage / carrier': 1, 'Backpack': 2, 'Bike handle': 3, 'Handheld': 4})

assert df['vehicle_rank'].notna().all()
assert df['carrying_contact_rank'].notna().all()
df[['Type of vehicle', 'vehicle_rank', 'Mode of carrying deliveries', 'carrying_contact_rank']].drop_duplicates()

,Type of vehicle,vehicle_rank,Mode of carrying deliveries,carrying_contact_rank
0,Scooter,1,Bike storage / carrier,1
1,Scooter,1,Handheld,4
2,Motor bike,2,Backpack,2
3,Motor bike,2,Bike handle,3
6,Motor bike,2,Handheld,4
10,Scooter,1,Backpack,2
14,Motor bike,2,Bike storage / carrier,1
18,Scooter,1,Bike handle,3


## Step 6 — Composite features

In [8]:
nasa_cols = [c for c in df.columns if c.endswith('[.]')]
borg_cols = [c for c in df.columns if c.endswith('[Your rating]')]
perf_col = [c for c in nasa_cols if 'satisfied' in c.lower()][0]
lifting_col = [c for c in borg_cols if 'lifting' in c.lower()][0]

# NASA-TLX mean, with the satisfaction item reversed (high satisfaction = low load)
nasa_adj = df[nasa_cols].copy()
nasa_adj[perf_col] = 100 - nasa_adj[perf_col]
df['workload_score'] = nasa_adj.mean(axis=1).round(1)
#workload_score = ( mental + physical + Time pressure + (100 - satisfied) + effort + stress ) / 6

df['fatigue_score'] = df[borg_cols].mean(axis=1).round(2)
#fatigue_score = ( overall + legs + breathing + lifting + traffic + exhaustion ) / 6

df['force_exertion'] = df[lifting_col]
#force_exertion = borg_lifting

df['vibration_index'] = df['vehicle_rank'] * df['work_hours_num']
#vibration_index = vehicle_rank * work_hours_num

df[['workload_score', 'fatigue_score', 'force_exertion', 'vibration_index']].describe()

,workload_score,fatigue_score,force_exertion,vibration_index
count,182.000000,182.000000,182.000000,182.000000
mean,50.478022,4.530549,4.148352,10.989011
std,14.915110,1.938360,2.645746,4.969508
min,16.700000,0.000000,0.000000,2.000000
25%,41.700000,3.170000,2.000000,7.000000
50%,50.000000,4.415000,4.000000,9.000000
75%,61.450000,5.957500,6.000000,18.000000
max,87.500000,9.000000,10.000000,18.000000


## Step 7 — Discomfort outcome (target)

In [9]:
# Outcome variable: any Nordic body-area pain in the last 12 months
nordic_cols = [c for c in df.columns if 'In the past 12 months' in c]
df['discomfort'] = (df[nordic_cols].sum(axis=1) > 0).astype(int)
df['discomfort'].value_counts().to_dict()

TypeError: '>' not supported between instances of 'str' and 'int'

## Step 8 — Validate and save

In [ ]:
assert df.isna().sum().sum() == 0
assert df.shape[0] == 182

out = ROOT / 'data' / 'processed' / 'model_ready.csv'
df.to_csv(out, index=False)
print('saved', out, df.shape)
df.dtypes.value_counts()

saved f:\Ergonomic_Project\data\processed\model_ready.csv (182, 67)


int64      45
object     18
float64     4
Name: count, dtype: int64